# This is my FYP model detection, improved version of P1 Defence. 
# By Mahnoor Bibi 

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json

ROOT = Path.cwd()

candidate_a = ROOT / "crime.v8i.yolov8"
candidate_b = ROOT / "models" / "weaponDetectionModel" / "crime.v8i.yolov8"

if candidate_a.exists():
    DATASET_ROOT = candidate_a
elif candidate_b.exists():
    DATASET_ROOT = candidate_b
else:
    DATASET_ROOT = candidate_a

SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}
CLASS_NAMES = {0: "criminal", 1: "person", 2: "weapon"}

print("Notebook CWD:", ROOT)
print("Dataset root:", DATASET_ROOT)
print("Exists:", DATASET_ROOT.exists())

# Count trainingi and validation data elements 

In [ ]:
split_inventory = {}

for split in SPLITS:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"

    images = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix in IMAGE_EXTS]) if img_dir.exists() else []
    labels = sorted(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

    split_inventory[split] = {
        "images": images,
        "labels": labels,
        "image_count": len(images),
        "label_count": len(labels),
    }

inventory_view = {
    s: {
        "image_count": split_inventory[s]["image_count"],
        "label_count": split_inventory[s]["label_count"],
    }
    for s in SPLITS
}

print(json.dumps(inventory_view, indent=2))
print("Total images:", sum(v["image_count"] for v in inventory_view.values()))

In [ ]:
box_counts = Counter()
image_presence = Counter()
malformed_rows = []
invalid_class_rows = []
invalid_range_rows = []
empty_label_files = []

for split in SPLITS:
    for label_path in split_inventory[split]["labels"]:
        lines = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()

        if not lines:
            empty_label_files.append((split, str(label_path)))
            continue

        classes_in_image = set()

        for line_no, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except Exception:
                malformed_rows.append((split, str(label_path), line_no, line))
                continue

            if cls not in CLASS_NAMES:
                invalid_class_rows.append((split, str(label_path), line_no, cls))
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                invalid_range_rows.append((split, str(label_path), line_no, (x, y, w, h)))
                continue

            box_counts[cls] += 1
            classes_in_image.add(cls)

        for cls in classes_in_image:
            image_presence[cls] += 1

summary_counts = {
    "box_counts": {CLASS_NAMES[k]: box_counts[k] for k in sorted(CLASS_NAMES)},
    "image_presence": {CLASS_NAMES[k]: image_presence[k] for k in sorted(CLASS_NAMES)},
    "empty_label_files": len(empty_label_files),
    "malformed_rows": len(malformed_rows),
    "invalid_class_rows": len(invalid_class_rows),
    "invalid_range_rows": len(invalid_range_rows),
}

print(json.dumps(summary_counts, indent=2))

# Balancing the dataset 
# Validating dataset if any invalid rows or column exist 
# bounding boxes count

In [ ]:
def imbalance_status(max_count, min_count):
    if min_count == 0:
        return "severe"
    ratio = max_count / min_count
    if ratio > 2.5:
        return "severe"
    if ratio > 1.5:
        return "mild"
    return "ok"

box_vals = [box_counts[k] for k in sorted(CLASS_NAMES)]
img_vals = [image_presence[k] for k in sorted(CLASS_NAMES)]

box_status = imbalance_status(max(box_vals), min(box_vals)) if box_vals else "n/a"
img_status = imbalance_status(max(img_vals), min(img_vals)) if img_vals else "n/a"

ratios = {
    "box_level_max_min_ratio": (max(box_vals) / min(box_vals)) if min(box_vals) > 0 else None,
    "image_level_max_min_ratio": (max(img_vals) / min(img_vals)) if min(img_vals) > 0 else None,
    "box_imbalance_status": box_status,
    "image_presence_imbalance_status": img_status,
}

print(json.dumps(ratios, indent=2))

In [ ]:
split_stats = {}
for split in SPLITS:
    n = split_inventory[split]["image_count"]
    split_stats[split] = {
        "count": n,
    }

total = sum(split_stats[s]["count"] for s in SPLITS)
for split in SPLITS:
    split_stats[split]["pct"] = round((split_stats[split]["count"] / total) * 100, 2) if total else 0.0

target = {
    "train_70": int(total * 0.70),
    "valid_20": int(total * 0.20),
    "test_10": total - int(total * 0.70) - int(total * 0.20),
}

print("Current split:")
print(json.dumps(split_stats, indent=2))
print("Target counts (70/20/10):")
print(json.dumps(target, indent=2))

# checking split percentage 

# our target is 70 20 10 
# train valid test 

# finally we have balance state mild : we will work with that

In [ ]:
def sha1_file(path: Path, chunk_size: int = 1 << 20):
    h = hashlib.sha1()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

hash_index = defaultdict(list)
stem_index = defaultdict(list)

for split in SPLITS:
    for img in split_inventory[split]["images"]:
        digest = sha1_file(img)
        hash_index[digest].append((split, str(img)))
        stem_index[img.stem.lower()].append((split, str(img)))

exact_leaks = [v for v in hash_index.values() if len({x[0] for x in v}) > 1]
stem_leaks = [v for v in stem_index.values() if len({x[0] for x in v}) > 1]

print("Exact cross-split duplicates:", len(exact_leaks))
print("Potential cross-split same-stem leaks:", len(stem_leaks))

if exact_leaks:
    print("Sample exact leak:", exact_leaks[0][:3])
if stem_leaks:
    print("Sample stem leak:", stem_leaks[0][:3])

In [ ]:
audit_report = {
    "dataset_root": str(DATASET_ROOT),
    "total_images": total,
    "split_counts": {s: split_stats[s]["count"] for s in SPLITS},
    "split_percentages": {s: split_stats[s]["pct"] for s in SPLITS},
    "target_70_20_10": target,
    "box_counts": {CLASS_NAMES[k]: box_counts[k] for k in sorted(CLASS_NAMES)},
    "image_presence_counts": {CLASS_NAMES[k]: image_presence[k] for k in sorted(CLASS_NAMES)},
    "quality_flags": {
        "empty_label_files": len(empty_label_files),
        "malformed_rows": len(malformed_rows),
        "invalid_class_rows": len(invalid_class_rows),
        "invalid_range_rows": len(invalid_range_rows),
    },
    "leakage_flags": {
        "exact_cross_split_duplicates": len(exact_leaks),
        "same_stem_cross_split_potential": len(stem_leaks),
    },
}

out_path = DATASET_ROOT / "phase1_audit_report.json"
out_path.write_text(json.dumps(audit_report, indent=2), encoding="utf-8")

print("Saved:", out_path)
print(json.dumps(audit_report, indent=2))

In [ ]:
import random
import math

report_path = DATASET_ROOT / "phase1_audit_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

box_counts_report = report.get("box_counts", {})
max_box = max(box_counts_report.values()) if box_counts_report else 0
min_box = min(box_counts_report.values()) if box_counts_report else 0
box_ratio = (max_box / min_box) if min_box else float("inf")

imbalance_state = "severe" if box_ratio > 2.5 else ("mild" if box_ratio > 1.5 else "ok")
proceed_annotation_audit = imbalance_state in {"ok", "mild"}

print("Imbalance state:", imbalance_state)
print("Proceed with annotation-quality phase:", proceed_annotation_audit)

# Random spot check sampling

In [ ]:
def parse_label_file(label_path: Path):
    rows = []
    lines = label_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    for i, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            continue
        try:
            cls = int(parts[0])
            x, y, w, h = map(float, parts[1:])
        except Exception:
            continue
        rows.append((i, cls, x, y, w, h))
    return rows


def to_xyxy(x, y, w, h):
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return (inter / union) if union > 0 else 0.0

In [ ]:
random.seed(42)

sample_total = 300
split_counts = report["split_counts"]

alloc = {}
remaining = sample_total
for idx, split in enumerate(SPLITS):
    if idx < len(SPLITS) - 1:
        n = round(sample_total * (split_counts[split] / report["total_images"]))
        alloc[split] = n
        remaining -= n
    else:
        alloc[split] = remaining

spotcheck_rows = []
for split in SPLITS:
    images = split_inventory[split]["images"]
    labels = {p.stem: p for p in split_inventory[split]["labels"]}
    k = min(alloc[split], len(images))
    chosen = random.sample(images, k) if k else []
    for img in chosen:
        lbl = labels.get(img.stem)
        anns = parse_label_file(lbl) if lbl and lbl.exists() else []
        cls_set = sorted({a[1] for a in anns})
        spotcheck_rows.append({
            "split": split,
            "image": str(img),
            "label": str(lbl) if lbl else "",
            "num_boxes": len(anns),
            "classes": [CLASS_NAMES.get(c, str(c)) for c in cls_set],
            "has_weapon": 2 in cls_set,
        })

spotcheck_path = DATASET_ROOT / "phase1_spotcheck_300.json"
spotcheck_path.write_text(json.dumps(spotcheck_rows, indent=2), encoding="utf-8")

print("Spot-check samples:", len(spotcheck_rows))
print("Saved:", spotcheck_path)
print("Per-split allocation:", alloc)

# checking for dublicates in dataset 
# checing for larger  bounding boexs 

In [ ]:
duplicate_overlap_flags = []
size_flags = []
keyword_missing_weapon_flags = []

weapon_keywords = ("gun", "knife", "pistol", "weapon", "rifle", "revolver")

for split in SPLITS:
    labels = split_inventory[split]["labels"]
    image_map = {p.stem: p for p in split_inventory[split]["images"]}

    for lbl in labels:
        anns = parse_label_file(lbl)
        if not anns:
            continue

        classes = [a[1] for a in anns]
        stem_lower = lbl.stem.lower()
        if any(k in stem_lower for k in weapon_keywords) and 2 not in classes:
            keyword_missing_weapon_flags.append({
                "split": split,
                "label": str(lbl),
                "image": str(image_map.get(lbl.stem, "")),
                "classes_present": sorted(set(classes)),
            })

        for idx, (_, cls, x, y, w, h) in enumerate(anns):
            area = w * h
            if area < 0.0005 or area > 0.80:
                size_flags.append({
                    "split": split,
                    "label": str(lbl),
                    "line_index": idx + 1,
                    "class_id": cls,
                    "bbox": [x, y, w, h],
                    "area": area,
                })

        for i in range(len(anns)):
            _, c1, x1, y1, w1, h1 = anns[i]
            b1 = to_xyxy(x1, y1, w1, h1)
            for j in range(i + 1, len(anns)):
                _, c2, x2, y2, w2, h2 = anns[j]
                if c1 != c2:
                    continue
                b2 = to_xyxy(x2, y2, w2, h2)
                ov = iou_xyxy(b1, b2)
                if ov > 0.9:
                    duplicate_overlap_flags.append({
                        "split": split,
                        "label": str(lbl),
                        "class_id": c1,
                        "pair": [i + 1, j + 1],
                        "iou": ov,
                    })

print("Duplicate overlap flags:", len(duplicate_overlap_flags))
print("Extreme bbox-size flags:", len(size_flags))
print("Keyword-based missing-weapon flags:", len(keyword_missing_weapon_flags))

In [ ]:
annotation_audit_report = {
    "proceed_annotation_audit": proceed_annotation_audit,
    "imbalance_state": imbalance_state,
    "spotcheck_sample_count": len(spotcheck_rows),
    "spotcheck_manifest": str(spotcheck_path),
    "flags_summary": {
        "duplicate_overlap_flags": len(duplicate_overlap_flags),
        "extreme_bbox_size_flags": len(size_flags),
        "keyword_missing_weapon_flags": len(keyword_missing_weapon_flags),
    },
    "flags_preview": {
        "duplicate_overlap_first10": duplicate_overlap_flags[:10],
        "extreme_bbox_size_first10": size_flags[:10],
        "keyword_missing_weapon_first10": keyword_missing_weapon_flags[:10],
    },
}

annotation_report_path = DATASET_ROOT / "phase1_annotation_quality_report.json"
annotation_report_path.write_text(json.dumps(annotation_audit_report, indent=2), encoding="utf-8")

print("Saved:", annotation_report_path)
print(json.dumps(annotation_audit_report["flags_summary"], indent=2))

# cheeckiing for images quality scan 
# total images ~ 7000

In [ ]:
import cv2
import numpy as np

all_split_images = []
for split in SPLITS:
    all_split_images.extend((split, p) for p in split_inventory[split]["images"])

print("Total images for quality scan:", len(all_split_images))

In [ ]:
def image_quality_metrics(img_bgr: np.ndarray):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(gray.mean())
    contrast = float(gray.std())
    h, w = gray.shape
    aspect_ratio = float(max(h, w) / max(1, min(h, w)))
    return {
        "h": int(h),
        "w": int(w),
        "lap_var": lap_var,
        "brightness": brightness,
        "contrast": contrast,
        "aspect_ratio": aspect_ratio,
    }

quality_rows = []
unreadable = []

for split, img_path in all_split_images:
    img = cv2.imread(str(img_path))
    if img is None:
        unreadable.append({"split": split, "image": str(img_path), "reason": "unreadable"})
        continue
    m = image_quality_metrics(img)
    quality_rows.append({
        "split": split,
        "image": str(img_path),
        **m,
    })

quality_path = DATASET_ROOT / "phase2_quality_metrics.json"
quality_path.write_text(json.dumps(quality_rows, indent=2), encoding="utf-8")

print("Saved:", quality_path)
print("Readable images:", len(quality_rows))
print("Unreadable images:", len(unreadable))

In [ ]:
HIGH_THRESHOLDS = {
    "very_dark": 20.0,
    "very_low_contrast": 10.0,
    "very_blurry": 18.0,
    "extreme_aspect_ratio": 3.5,
}

MEDIUM_THRESHOLDS = {
    "dark": 35.0,
    "low_contrast": 18.0,
    "blurry": 40.0,
}

high_noise = []
medium_noise = []

for row in quality_rows:
    reasons_high = []
    reasons_medium = []

    if row["brightness"] < HIGH_THRESHOLDS["very_dark"] and row["contrast"] < HIGH_THRESHOLDS["very_low_contrast"]:
        reasons_high.append("very_dark_and_low_contrast")
    if row["lap_var"] < HIGH_THRESHOLDS["very_blurry"] and row["contrast"] < HIGH_THRESHOLDS["very_low_contrast"]:
        reasons_high.append("very_blurry_and_low_contrast")
    if row["aspect_ratio"] > HIGH_THRESHOLDS["extreme_aspect_ratio"]:
        reasons_high.append("extreme_aspect_ratio")

    if row["brightness"] < MEDIUM_THRESHOLDS["dark"]:
        reasons_medium.append("dark")
    if row["contrast"] < MEDIUM_THRESHOLDS["low_contrast"]:
        reasons_medium.append("low_contrast")
    if row["lap_var"] < MEDIUM_THRESHOLDS["blurry"]:
        reasons_medium.append("blurry")

    if reasons_high:
        high_noise.append({**row, "reasons": sorted(set(reasons_high))})
    elif len(set(reasons_medium)) >= 2:
        medium_noise.append({**row, "reasons": sorted(set(reasons_medium))})

if unreadable:
    for bad in unreadable:
        high_noise.append({
            "split": bad["split"],
            "image": bad["image"],
            "h": None,
            "w": None,
            "lap_var": None,
            "brightness": None,
            "contrast": None,
            "aspect_ratio": None,
            "reasons": ["unreadable"],
        })

noise_report = {
    "high_noise_count": len(high_noise),
    "medium_noise_count": len(medium_noise),
    "high_noise_preview": high_noise[:30],
    "medium_noise_preview": medium_noise[:30],
}

noise_report_path = DATASET_ROOT / "phase2_noise_report.json"
noise_report_path.write_text(json.dumps(noise_report, indent=2), encoding="utf-8")

high_list_path = DATASET_ROOT / "phase2_high_noise_candidates.json"
high_list_path.write_text(json.dumps(high_noise, indent=2), encoding="utf-8")

print("Saved:", noise_report_path)
print("Saved:", high_list_path)
print(json.dumps({"high_noise_count": len(high_noise), "medium_noise_count": len(medium_noise)}, indent=2))

In [ ]:
from shutil import move

APPLY_HIGH_NOISE_QUARANTINE = False

quarantine_root = DATASET_ROOT / "_quarantine_high_noise"

moved = []
missing_pairs = []

for row in high_noise:
    img = Path(row["image"])
    split = row["split"]
    lbl = DATASET_ROOT / split / "labels" / f"{img.stem}.txt"

    q_img_dir = quarantine_root / split / "images"
    q_lbl_dir = quarantine_root / split / "labels"

    if APPLY_HIGH_NOISE_QUARANTINE:
        q_img_dir.mkdir(parents=True, exist_ok=True)
        q_lbl_dir.mkdir(parents=True, exist_ok=True)

        if img.exists():
            move(str(img), str(q_img_dir / img.name))
        if lbl.exists():
            move(str(lbl), str(q_lbl_dir / lbl.name))
        else:
            missing_pairs.append(str(lbl))

    moved.append({
        "split": split,
        "image": str(img),
        "label": str(lbl),
        "reasons": row["reasons"],
    })

quarantine_manifest = {
    "apply_high_noise_quarantine": APPLY_HIGH_NOISE_QUARANTINE,
    "high_noise_candidates": len(high_noise),
    "medium_noise_kept": len(medium_noise),
    "moved_or_planned": len(moved),
    "missing_labels_when_apply_true": len(missing_pairs),
    "items_preview": moved[:40],
}

quarantine_manifest_path = DATASET_ROOT / "phase2_high_noise_quarantine_manifest.json"
quarantine_manifest_path.write_text(json.dumps(quarantine_manifest, indent=2), encoding="utf-8")

print("Saved:", quarantine_manifest_path)
print(json.dumps({
    "apply_high_noise_quarantine": APPLY_HIGH_NOISE_QUARANTINE,
    "high_noise_candidates": len(high_noise),
    "medium_noise_kept": len(medium_noise),
}, indent=2))

# filtering images based on given thresholds 

LOW_VALUE_THRESHOLDS = {
    "closeup_area_high": 0.65,
    "closeup_area_extreme": 0.80,
    "posterized_unique_gray_ratio": 0.05,
    "oversaturated_mean_sat": 185.0,
    "overexposed_mean_brightness": 245.0,
    "underexposed_mean_brightness": 8.0,
    "high_contrast_std": 95.0,
    "flat_contrast_std": 6.0,
}


In [ ]:
import cv2
import numpy as np
from pathlib import Path

CARTOON_NAME_KEYWORDS = {
    "cartoon", "anime", "illustration", "illustrated", "drawing", "sketch",
    "clipart", "vector", "render", "cgi", "3d-art", "graphic"
}

LOW_VALUE_THRESHOLDS = {
    "closeup_area_high": 0.65,
    "closeup_area_extreme": 0.80,
    "posterized_unique_gray_ratio": 0.05,
    "oversaturated_mean_sat": 185.0,
    "overexposed_mean_brightness": 245.0,
    "underexposed_mean_brightness": 8.0,
    "high_contrast_std": 95.0,
    "flat_contrast_std": 6.0,
}


def image_style_metrics(img_bgr: np.ndarray):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    unique_gray = np.unique(gray)
    unique_gray_ratio = float(len(unique_gray) / 256.0)

    mean_sat = float(hsv[:, :, 1].mean())
    mean_val = float(hsv[:, :, 2].mean())
    contrast_std = float(gray.std())

    return {
        "unique_gray_ratio": unique_gray_ratio,
        "mean_saturation": mean_sat,
        "mean_brightness": mean_val,
        "contrast_std": contrast_std,
    }


def max_box_area_and_classes(label_path: Path):
    anns = parse_label_file(label_path)
    if not anns:
        return 0.0, set(), 0
    areas = []
    classes = set()
    for _, cls, _x, _y, w, h in anns:
        areas.append(float(w * h))
        classes.add(cls)
    return (max(areas) if areas else 0.0), classes, len(anns)

In [ ]:
low_value_high_conf = []
low_value_review = []

for split in SPLITS:
    img_map = {p.stem: p for p in split_inventory[split]["images"]}
    lbl_map = {p.stem: p for p in split_inventory[split]["labels"]}

    for stem, img_path in img_map.items():
        label_path = lbl_map.get(stem)
        if label_path is None:
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        style = image_style_metrics(img)
        max_area, classes, n_boxes = max_box_area_and_classes(label_path)
        stem_lower = stem.lower()

        flags_high = []
        flags_review = []

        # A) Cartoons / illustrations
        if any(k in stem_lower for k in CARTOON_NAME_KEYWORDS):
            flags_high.append("filename_cartoon_or_illustration")
        elif (
            style["unique_gray_ratio"] < LOW_VALUE_THRESHOLDS["posterized_unique_gray_ratio"]
            and style["mean_saturation"] > LOW_VALUE_THRESHOLDS["oversaturated_mean_sat"]
        ):
            flags_review.append("possible_illustration_style")

        # B) Unrealistic studio-only closeups
        if max_area >= LOW_VALUE_THRESHOLDS["closeup_area_extreme"] and classes.issubset({2}):
            flags_high.append("extreme_weapon_closeup_no_context")
        elif max_area >= LOW_VALUE_THRESHOLDS["closeup_area_high"] and classes.issubset({2}):
            flags_review.append("weapon_closeup_low_context")

        # C) Extreme edits not representative of deployment
        if (
            style["mean_brightness"] >= LOW_VALUE_THRESHOLDS["overexposed_mean_brightness"]
            and style["contrast_std"] <= LOW_VALUE_THRESHOLDS["flat_contrast_std"]
        ):
            flags_high.append("overexposed_flat_extreme")
        elif (
            style["mean_brightness"] <= LOW_VALUE_THRESHOLDS["underexposed_mean_brightness"]
            and style["contrast_std"] >= LOW_VALUE_THRESHOLDS["high_contrast_std"]
        ):
            flags_high.append("underexposed_harsh_extreme")

        if flags_high:
            low_value_high_conf.append({
                "split": split,
                "image": str(img_path),
                "label": str(label_path),
                "flags": sorted(set(flags_high)),
                "style": style,
                "max_box_area": max_area,
                "classes": sorted(list(classes)),
                "n_boxes": n_boxes,
            })
        elif flags_review:
            low_value_review.append({
                "split": split,
                "image": str(img_path),
                "label": str(label_path),
                "flags": sorted(set(flags_review)),
                "style": style,
                "max_box_area": max_area,
                "classes": sorted(list(classes)),
                "n_boxes": n_boxes,
            })

print("High-confidence low-value candidates:", len(low_value_high_conf))
print("Review-only candidates (kept):", len(low_value_review))

In [ ]:
low_value_report = {
    "high_confidence_candidates": len(low_value_high_conf),
    "review_only_candidates": len(low_value_review),
    "high_confidence_preview": low_value_high_conf[:50],
    "review_only_preview": low_value_review[:50],
}

low_value_candidates_path = DATASET_ROOT / "phase2_low_value_candidates.json"
low_value_report_path = DATASET_ROOT / "phase2_low_value_report.json"

low_value_candidates_path.write_text(json.dumps(low_value_high_conf, indent=2), encoding="utf-8")
low_value_report_path.write_text(json.dumps(low_value_report, indent=2), encoding="utf-8")

print("Saved:", low_value_candidates_path)
print("Saved:", low_value_report_path)

# GRouping into valid, testing, trainging

In [ ]:
import re
import random
import shutil

random.seed(42)


def source_key_from_stem(stem: str):
    return stem.split('.rf.')[0]


def sequence_key_from_source(source_key: str):
    key = source_key.lower()
    key = re.sub(r'(frame)\d+$', r'\1', key)
    key = re.sub(r'(mp4-)\d+$', r'\1', key)
    key = re.sub(r'(video-)\d+$', r'\1', key)
    return key

records = []
for split in SPLITS:
    img_map = {p.stem: p for p in split_inventory[split]["images"]}
    lbl_map = {p.stem: p for p in split_inventory[split]["labels"]}
    for stem, img_path in img_map.items():
        lbl_path = lbl_map.get(stem)
        if lbl_path is None:
            continue
        src_key = source_key_from_stem(stem)
        seq_key = sequence_key_from_source(src_key)
        records.append({
            "orig_split": split,
            "stem": stem,
            "source_key": src_key,
            "sequence_key": seq_key,
            "img": img_path,
            "lbl": lbl_path,
        })

print("Paired image+label records:", len(records))

In [ ]:
groups = defaultdict(list)
for r in records:
    groups[r["sequence_key"]].append(r)

group_items = list(groups.items())
random.shuffle(group_items)

total = len(records)
target_counts = {
    "train": int(round(total * 0.70)),
    "valid": int(round(total * 0.20)),
    "test": total - int(round(total * 0.70)) - int(round(total * 0.20)),
}

assigned = {"train": [], "valid": [], "test": []}
current = {"train": 0, "valid": 0, "test": 0}

for gkey, items in group_items:
    size = len(items)
    deficits = {s: target_counts[s] - current[s] for s in ["train", "valid", "test"]}
    best_split = max(deficits, key=deficits.get)
    assigned[best_split].extend(items)
    current[best_split] += size

resplit_counts = {s: len(assigned[s]) for s in ["train", "valid", "test"]}
print("Target:", target_counts)
print("Assigned:", resplit_counts)
print("Total assigned:", sum(resplit_counts.values()))

In [ ]:
def sha1_file(path: Path, chunk_size: int = 1 << 20):
    h = hashlib.sha1()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

new_hash_map = defaultdict(set)
new_source_map = defaultdict(set)
new_sequence_map = defaultdict(set)

for split in ["train", "valid", "test"]:
    for r in assigned[split]:
        digest = sha1_file(r["img"])
        new_hash_map[digest].add(split)
        new_source_map[r["source_key"]].add(split)
        new_sequence_map[r["sequence_key"]].add(split)

hash_leaks = [k for k, v in new_hash_map.items() if len(v) > 1]
source_leaks = [k for k, v in new_source_map.items() if len(v) > 1]
sequence_leaks = [k for k, v in new_sequence_map.items() if len(v) > 1]

print("Cross-split exact hash leaks:", len(hash_leaks))
print("Cross-split source-key leaks:", len(source_leaks))
print("Cross-split sequence-key leaks:", len(sequence_leaks))

In [ ]:
resplit_report = {
    "total_records": len(records),
    "target_counts_70_20_10": target_counts,
    "assigned_counts": resplit_counts,
    "leakage_checks": {
        "exact_hash_cross_split": len(hash_leaks),
        "source_key_cross_split": len(source_leaks),
        "sequence_key_cross_split": len(sequence_leaks),
    },
}

resplit_report_path = DATASET_ROOT / "phase1_resplit_report.json"
resplit_report_path.write_text(json.dumps(resplit_report, indent=2), encoding="utf-8")

assignment_manifest = {
    split: [
        {
            "image": str(r["img"]),
            "label": str(r["lbl"]),
            "source_key": r["source_key"],
            "sequence_key": r["sequence_key"],
        }
        for r in assigned[split]
    ]
    for split in ["train", "valid", "test"]
}

assignment_manifest_path = DATASET_ROOT / "phase1_resplit_assignment_manifest.json"
assignment_manifest_path.write_text(json.dumps(assignment_manifest, indent=2), encoding="utf-8")

print("Saved:", resplit_report_path)
print("Saved:", assignment_manifest_path)
print(json.dumps(resplit_report, indent=2))

In [ ]:
APPLY_RESPLIT_DATASET = False

resplit_root = DATASET_ROOT.parent / f"{DATASET_ROOT.name}_resplit_70_20_10"

if APPLY_RESPLIT_DATASET:
    if resplit_root.exists():
        shutil.rmtree(resplit_root)

    for split in ["train", "valid", "test"]:
        (resplit_root / split / "images").mkdir(parents=True, exist_ok=True)
        (resplit_root / split / "labels").mkdir(parents=True, exist_ok=True)

    for split in ["train", "valid", "test"]:
        for r in assigned[split]:
            shutil.copy2(r["img"], resplit_root / split / "images" / r["img"].name)
            shutil.copy2(r["lbl"], resplit_root / split / "labels" / r["lbl"].name)

    data_yaml_src = DATASET_ROOT / "data.yaml"
    if data_yaml_src.exists():
        text = data_yaml_src.read_text(encoding="utf-8")
        text = re.sub(r"train:\s*.*", "train: ../train/images", text)
        text = re.sub(r"val:\s*.*", "val: ../valid/images", text)
        text = re.sub(r"test:\s*.*", "test: ../test/images", text)
        (resplit_root / "data.yaml").write_text(text, encoding="utf-8")

print("APPLY_RESPLIT_DATASET:", APPLY_RESPLIT_DATASET)
print("Target resplit folder:", resplit_root)

In [ ]:
import json
from pathlib import Path

PHASE2_ROOT = DATASET_ROOT

phase2_profile = {
    "input_normalization": {
        "primary_train_imgsz": 640,
        "secondary_edge_imgsz": 416,
        "recommended_stride_alignment": 32,
        "letterbox_mode": "fit",
    },
    "class_schema": {
        "num_classes": 3,
        "class_names": ["criminal", "person", "weapon"],
        "valid_class_ids": [0, 1, 2],
    },
}

phase2_profile_path = PHASE2_ROOT / "phase2_preprocessing_profile.json"
phase2_profile_path.write_text(json.dumps(phase2_profile, indent=2), encoding="utf-8")

print("Saved:", phase2_profile_path)
print(json.dumps(phase2_profile, indent=2))

# Data annotations phase 
# data augmentation phase
# from splitted data 

In [ ]:
from collections import Counter

annotation_issues = {
    "missing_label_file": [],
    "malformed_row": [],
    "invalid_class_id": [],
    "invalid_range": [],
}

box_count = 0
class_counts = Counter()
file_count = 0

for split in ["train", "valid", "test"]:
    img_dir = PHASE2_ROOT / split / "images"
    lbl_dir = PHASE2_ROOT / split / "labels"

    images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}] if img_dir.exists() else []

    for img in images:
        lbl = lbl_dir / f"{img.stem}.txt"
        if not lbl.exists():
            annotation_issues["missing_label_file"].append({"split": split, "image": str(img), "expected_label": str(lbl)})
            continue

        file_count += 1
        lines = lbl.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
        for line_no, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                annotation_issues["malformed_row"].append({"split": split, "label": str(lbl), "line": line_no, "raw": line})
                continue

            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except Exception:
                annotation_issues["malformed_row"].append({"split": split, "label": str(lbl), "line": line_no, "raw": line})
                continue

            if cls not in {0, 1, 2}:
                annotation_issues["invalid_class_id"].append({"split": split, "label": str(lbl), "line": line_no, "class_id": cls})
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                annotation_issues["invalid_range"].append({
                    "split": split,
                    "label": str(lbl),
                    "line": line_no,
                    "bbox": [x, y, w, h],
                })
                continue

            class_counts[cls] += 1
            box_count += 1

phase2_annotation_report = {
    "checked_label_files": file_count,
    "total_valid_boxes": box_count,
    "class_counts": {"criminal": class_counts[0], "person": class_counts[1], "weapon": class_counts[2]},
    "issues_summary": {k: len(v) for k, v in annotation_issues.items()},
    "issues_preview": {k: v[:25] for k, v in annotation_issues.items()},
}

phase2_annotation_report_path = PHASE2_ROOT / "phase2_annotation_verification.json"
phase2_annotation_report_path.write_text(json.dumps(phase2_annotation_report, indent=2), encoding="utf-8")

print("Saved:", phase2_annotation_report_path)
print(json.dumps(phase2_annotation_report["issues_summary"], indent=2))

In [ ]:
phase2_augmentation_profile = {
    "intent": "realistic_cctv",
    "allowed_augmentations": {
        "hsv_h": 0.01,
        "hsv_s": 0.35,
        "hsv_v": 0.30,
        "translate": 0.08,
        "scale": 0.20,
        "perspective": 0.0008,
        "fliplr": 0.5,
        "flipud": 0.0,
        "mosaic": 0.15,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "erasing": 0.15,
        "close_mosaic_epoch": 10,
    },
    "cctv_effects_post_yolo": {
        "gaussian_noise_prob": 0.25,
        "jpeg_artifact_prob": 0.25,
        "motion_blur_prob": 0.20,
        "low_light_prob": 0.30,
        "partial_occlusion_prob": 0.20,
    },
    "blocked_aggressive_transforms": [
        "large_perspective_warp",
        "heavy_rotation",
        "semantic_altering_cutmix",
        "synthetic_object_paste"
    ],
}

phase2_aug_profile_path = PHASE2_ROOT / "phase2_augmentation_profile.json"
phase2_aug_profile_path.write_text(json.dumps(phase2_augmentation_profile, indent=2), encoding="utf-8")

print("Saved:", phase2_aug_profile_path)
print(json.dumps(phase2_augmentation_profile, indent=2))

In [ ]:
quality_metrics_path = PHASE2_ROOT / "phase2_quality_metrics.json"
quality_rows = json.loads(quality_metrics_path.read_text(encoding="utf-8")) if quality_metrics_path.exists() else []
quality_by_image = {row["image"]: row for row in quality_rows}

hard_far_distance = []
hard_partial_visibility = []
hard_crowd_multi_person = []
hard_reflection_clutter = []

for split in ["train", "valid", "test"]:
    lbl_dir = PHASE2_ROOT / split / "labels"
    img_dir = PHASE2_ROOT / split / "images"
    for lbl in lbl_dir.glob("*.txt"):
        img = img_dir / f"{lbl.stem}.jpg"
        if not img.exists():
            jpeg = img_dir / f"{lbl.stem}.jpeg"
            png = img_dir / f"{lbl.stem}.png"
            img = jpeg if jpeg.exists() else (png if png.exists() else img)

        anns = parse_label_file(lbl)
        if not anns:
            continue

        person_count = 0
        weapon_boxes = []
        edge_touch_weapon = False

        for _, cls, x, y, w, h in anns:
            if cls == 1:
                person_count += 1
            if cls == 2:
                area = w * h
                weapon_boxes.append(area)
                x1, y1, x2, y2 = to_xyxy(x, y, w, h)
                if x1 < 0.05 or y1 < 0.05 or x2 > 0.95 or y2 > 0.95:
                    edge_touch_weapon = True

        if weapon_boxes and min(weapon_boxes) < 0.01:
            hard_far_distance.append({"split": split, "image": str(img), "label": str(lbl), "min_weapon_area": min(weapon_boxes)})

        if weapon_boxes and edge_touch_weapon:
            hard_partial_visibility.append({"split": split, "image": str(img), "label": str(lbl)})

        if person_count >= 3 and weapon_boxes:
            hard_crowd_multi_person.append({
                "split": split,
                "image": str(img),
                "label": str(lbl),
                "person_count": person_count,
                "weapon_count": len(weapon_boxes),
            })

        q = quality_by_image.get(str(img))
        if q is not None:
            clutter_signal = (q.get("contrast", 0) >= 60 and q.get("lap_var", 0) >= 120)
            if clutter_signal and weapon_boxes:
                hard_reflection_clutter.append({"split": split, "image": str(img), "label": str(lbl)})

# checkign status of final dataset 

In [ ]:
def dedupe_by_image(rows):
    seen = set()
    out = []
    for r in rows:
        key = r["image"]
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out

hard_far_distance = dedupe_by_image(hard_far_distance)
hard_partial_visibility = dedupe_by_image(hard_partial_visibility)
hard_crowd_multi_person = dedupe_by_image(hard_crowd_multi_person)
hard_reflection_clutter = dedupe_by_image(hard_reflection_clutter)

MAX_PER_BUCKET = 150
curated_hard_set = {
    "far_distance_weapons": hard_far_distance[:MAX_PER_BUCKET],
    "partial_visibility": hard_partial_visibility[:MAX_PER_BUCKET],
    "crowd_multi_person": hard_crowd_multi_person[:MAX_PER_BUCKET],
    "reflection_clutter": hard_reflection_clutter[:MAX_PER_BUCKET],
}

phase2_hard_examples = {
    "counts": {k: len(v) for k, v in curated_hard_set.items()},
    "items": curated_hard_set,
}

phase2_hard_path = PHASE2_ROOT / "phase2_hard_example_set.json"
phase2_hard_path.write_text(json.dumps(phase2_hard_examples, indent=2), encoding="utf-8")

print("Saved:", phase2_hard_path)
print(json.dumps(phase2_hard_examples["counts"], indent=2))

In [ ]:
phase2_final_status = {
    "preprocessing_profile_file": str(phase2_profile_path),
    "annotation_verification_file": str(phase2_annotation_report_path),
    "augmentation_profile_file": str(phase2_aug_profile_path),
    "hard_example_set_file": str(phase2_hard_path),
    "ready_for_training": (
        phase2_annotation_report["issues_summary"].get("malformed_row", 0) == 0
        and phase2_annotation_report["issues_summary"].get("invalid_class_id", 0) == 0
        and phase2_annotation_report["issues_summary"].get("invalid_range", 0) == 0
    ),
}

phase2_status_path = PHASE2_ROOT / "phase2_status.json"
phase2_status_path.write_text(json.dumps(phase2_final_status, indent=2), encoding="utf-8")

print("Saved:", phase2_status_path)
print(json.dumps(phase2_final_status, indent=2))

# -------- Phase 3 starts here -------- 

In [ ]:
PHASE3_CONFIG = {
    "v8n": {
        "base_weights": "yolov8n.pt",
        "batch": 4,
        "imgsz": 640,
        "epochs_stage_a": 25,
        "epochs_stage_b": 25,
        "patience": 15,
    },
    "v8s": {
        "base_weights": "yolov8s.pt",
        "batch": 2,
        "imgsz": 640,
        "epochs_stage_a": 25,
        "epochs_stage_b": 25,
        "patience": 15,
    },
}

print(json.dumps(PHASE3_CONFIG, indent=2))

In [ ]:
def train_two_stage(model_key: str, cfg: dict):
    run_name = f"phase3_{model_key}"

    model = YOLO(cfg["base_weights"])
    model.train(
        data=DATA_YAML,
        epochs=cfg["epochs_stage_a"],
        imgsz=cfg["imgsz"],
        batch=cfg["batch"],
        device=DEVICE,
        workers=4,
        project=str(PHASE3_OUT),
        name=run_name,
        exist_ok=True,
        amp=True,
        pretrained=True,
        patience=cfg["patience"],
        optimizer="auto",
    )

    last_pt = PHASE3_OUT / run_name / "weights" / "last.pt"
    resume_model = YOLO(str(last_pt))
    resume_model.train(
        data=DATA_YAML,
        epochs=cfg["epochs_stage_b"],
        imgsz=cfg["imgsz"],
        batch=cfg["batch"],
        device=DEVICE,
        workers=4,
        project=str(PHASE3_OUT),
        name=run_name,
        exist_ok=True,
        amp=True,
        resume=True,
        patience=cfg["patience"],
        optimizer="auto",
    )

    run_dir = PHASE3_OUT / run_name
    return {
        "run_name": run_name,
        "run_dir": str(run_dir),
        "results_csv": str(run_dir / "results.csv"),
        "best_pt": str(run_dir / "weights" / "best.pt"),
        "last_pt": str(run_dir / "weights" / "last.pt"),
        "confusion_matrix": str(run_dir / "confusion_matrix.png"),
    }

In [ ]:
# Execute remaining training stages (run only what is still pending).
# v8n Stage B: continue from Stage A best model.
v8n_stageA_best = PHASE3_OUT / "phase3_v8n" / "weights" / "best.pt"
if v8n_stageA_best.exists() and not (PHASE3_OUT / "phase3_v8n_stageB" / "results.csv").exists():
    _ = run_stage(v8n_stageA_best, "phase3_v8n_stageB", epochs=25, batch=2, imgsz=640, amp=True, workers=2)

# v8s Stage A + B (safer default for 4GB VRAM).
v8s_stageA_best = PHASE3_OUT / "phase3_v8s" / "weights" / "best.pt"
if not (PHASE3_OUT / "phase3_v8s" / "results.csv").exists():
    _ = run_stage("yolov8s.pt", "phase3_v8s", epochs=25, batch=2, imgsz=640, amp=True, workers=2)

if v8s_stageA_best.exists() and not (PHASE3_OUT / "phase3_v8s_stageB" / "results.csv").exists():
    _ = run_stage(v8s_stageA_best, "phase3_v8s_stageB", epochs=25, batch=2, imgsz=640, amp=True, workers=2)

phase3_runs = {}
for run_name in ["phase3_v8n", "phase3_v8n_stageB", "phase3_v8s", "phase3_v8s_stageB"]:
    info = discover_run(run_name)
    if info:
        phase3_runs[run_name] = info

phase3_runs_path = PHASE3_OUT / "phase3_run_artifacts_v2.json"
phase3_runs_path.write_text(json.dumps(phase3_runs, indent=2), encoding="utf-8")
print("Saved:", phase3_runs_path)
print(json.dumps(phase3_runs, indent=2))

# Due to GPU memory full error next 25 epochs 

In [ ]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path("/home/alpha/Desktop/FYP/models/weaponDetectionModel/crime.v8i.yolov8/phase3_outputs/phase3_v8n_stageB")
last_ckpt = run_dir / "weights" / "last.pt"

assert last_ckpt.exists(), f"Missing checkpoint: {last_ckpt}"

model = YOLO(str(last_ckpt))
model.train(resume=True)

# Trainign completed 

In [ ]:
# metrics 
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
DATASET_ROOT = ROOT / "crime.v8i.yolov8"
if not DATASET_ROOT.exists():
    DATASET_ROOT = ROOT / "models" / "weaponDetectionModel" / "crime.v8i.yolov8"

PHASE3_OUT = DATASET_ROOT / "phase3_outputs"

RUNS = {
    "v8n_stageB": PHASE3_OUT / "phase3_v8n_stageB",
    "v8s_stageB": PHASE3_OUT / "phase3_v8s_stageB",
}

KEY_COLS = [
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
]

histories = {}
summary_rows = []

for run_name, run_dir in RUNS.items():
    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        print(f"[WARN] Missing results: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    histories[run_name] = df

    row = {"run": run_name}
    if "epoch" in df.columns and not df["epoch"].dropna().empty:
        row["epochs_logged"] = int(df["epoch"].max()) + 1
    else:
        row["epochs_logged"] = len(df)

    for col in KEY_COLS:
        if col in df.columns and not df[col].dropna().empty:
            if col.startswith("metrics/"):
                row[f"best::{col}"] = float(df[col].max())
            else:
                row[f"final::{col}"] = float(df[col].dropna().iloc[-1])

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("Evaluation summary:")
display(summary_df)

summary_out = PHASE3_OUT / "phase4_eval_summary.json"
summary_out.write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
print("Saved:", summary_out)

In [ ]:
# key training curves.
import matplotlib.pyplot as plt

if not histories:
    raise RuntimeError("No run histories loaded. Run the previous evaluation cell first.")

for run_name, df in histories.items():
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    if "metrics/mAP50(B)" in df.columns:
        axes[0, 0].plot(df["metrics/mAP50(B)"], label="mAP@0.5")
    if "metrics/mAP50-95(B)" in df.columns:
        axes[0, 0].plot(df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
    axes[0, 0].set_title(f"{run_name} - mAP")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()

    if "metrics/precision(B)" in df.columns:
        axes[0, 1].plot(df["metrics/precision(B)"], label="Precision")
    if "metrics/recall(B)" in df.columns:
        axes[0, 1].plot(df["metrics/recall(B)"], label="Recall")
    axes[0, 1].set_title(f"{run_name} - Precision/Recall")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()

    if "train/box_loss" in df.columns:
        axes[1, 0].plot(df["train/box_loss"], label="train box")
    if "val/box_loss" in df.columns:
        axes[1, 0].plot(df["val/box_loss"], label="val box")
    axes[1, 0].set_title(f"{run_name} - Box Loss")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()

    if "train/cls_loss" in df.columns:
        axes[1, 1].plot(df["train/cls_loss"], label="train cls")
    if "val/cls_loss" in df.columns:
        axes[1, 1].plot(df["val/cls_loss"], label="val cls")
    axes[1, 1].set_title(f"{run_name} - Cls Loss")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].legend()

    plt.tight_layout()
    out = PHASE3_OUT / f"phase4_{run_name}_curves.png"
    plt.savefig(out, dpi=160)
    plt.show()
    print("Saved:", out)